In [107]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)
documents = [file.parse() for file in reader.read()]

In [108]:
data_gen_instructions = """
You emulate a student who is taking our LLM course.
You are given one lesson page from the course.
Formulate 5 questions this student might ask that are answered by this page.

Rules:
- The page should contain the answer to each question.
- Make the questions complete and not too short.
- Use as few words as possible from the page; don't copy its phrasing.
- The questions should resemble how people actually ask things online:
  not too formal, not too short, not too long.
- Ask about the content of the lesson, not about its formatting or filename.
""".strip()

General OpenAI API configuration

In [109]:
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
openai_client = OpenAI()

Model for questions

In [110]:
from pydantic import BaseModel

class Questions(BaseModel):
    questions: list[str]

## Q1. Generating questions

In [111]:
q1_documents = documents[:3]
q1_documents

[{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuation.\nYour phone uses a sim

Importing ground truth generation

In [112]:
from evaluation_utils import llm_structured_retry # llm_structured_retry wraps the same call in a retry loop.
import json

def generate_ground_truth(doc):
    user_prompt = json.dumps(doc)

    out, usage = llm_structured_retry(
        openai_client,
        data_gen_instructions,
        user_prompt,
        Questions
    )

    results = []

    for q in out.questions:
        results.append({
            "question": q,
            "document": doc["filename"]
        })

    return results, usage

Get results

In [113]:
from tqdm.auto import tqdm

ground_truth = []
usages = []

for doc in tqdm(q1_documents):
    records, usage = generate_ground_truth(doc)
    ground_truth.extend(records)
    usages.append(usage)

  0%|          | 0/3 [00:00<?, ?it/s]

In [114]:
ground_truth

[{'question': "What is RAG, and how does it help when an LLM doesn't know the answer by itself?",
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'Why does the lesson say LLMs are treated like a black box in this course?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What are some main limits of LLMs that RAG is meant to fix?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': "What are the main things you'll build in the first part of this module?",
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'How is the second part of the module different from the first one?',
  'document': '01-agentic-rag/lessons/01-intro.md'},
 {'question': 'What do I need installed before starting this module, and can I use only Jupyter with Python?',
  'document': '01-agentic-rag/lessons/02-environment.md'},
 {'question': 'How do I set up a new project from zero for this course, and which packages should I add?',
  'document': '01-a

Costs

In [115]:
(usages[0].input_tokens + usages[1].input_tokens + usages[2].input_tokens)/3

1353.0

## Q2. First result with text search

Preparatives

In [116]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)
len(chunks)

295

Define rrf function

In [117]:
def rrf(result_lists, k=60, num_results=5):
    scores = {}
    docs = {}

    for results in result_lists:
        for rank, doc in enumerate(results):
            key = (doc["filename"], doc.get("start", rank))
            scores[key] = scores.get(key, 0) + 1 / (k + rank)
            docs[key] = doc

    ranked = sorted(scores, key=scores.get, reverse=True)
    return [docs[key] for key in ranked[:num_results]]


Hybrid search

In [118]:

import numpy as np
from minsearch import VectorSearch, Index

from ingest import build_index
from embedder import Embedder


index = build_index(documents)
embedder = Embedder()

encoded_chunks = [embedder.encode(chunk["content"]) for chunk in chunks]
X = np.array(encoded_chunks)

vindex = VectorSearch(keyword_fields=['filename'])
vindex.fit(
    vectors=X,
    payload=chunks,
)


def text_search(query, num_results=5):
    boost_dict = {"content": 3.0, "filename": 0.5}

    return index.search(
        query,
        num_results=num_results,
        boost_dict=boost_dict,
    )


def vector_search(query, num_results=5):
    query_vector = embedder.encode(query)
    return vindex.search(query_vector, num_results=num_results)


In [119]:
def hybrid_search(query, k=60):
    text_results = text_search(query, num_results=10)
    vector_results = vector_search(query, num_results=10)
    return rrf([text_results, vector_results], k=k)

First question from the ground truth

In [120]:
q = ground_truth[0]["question"]
q

"What is RAG, and how does it help when an LLM doesn't know the answer by itself?"

In [121]:
text_results = text_search(q)
text_results

[{'content': '# RAG\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=JktYwBIDErk&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nWe run free Zoomcamp courses at DataTalks.Club on data engineering,\nmachine learning, MLOps, and other topics. Each course has its own\nFAQ document with common questions and answers.\n\nSome of these documents have over 300 questions. Students ask us\nthings in Slack like "Can I still join after the course started?" or\n"How do I get a certificate?" Finding those answers in the FAQ is\ntedious.\n\nWe want a bot that takes all this knowledge and answers student\nquestions in natural language.\n\nIn this module, we\'ll build that system. But first, let\'s see why we\ncan\'t send the question straight to an LLM and call it a day.\n\n## Plain LLMs lack our data\n\nFirst, let\'s define a function to talk to the LLM:\n\n```python\ndef llm(prompt):\n    response = openai_client.responses.create(\n        model="gpt-5.4-mini",\n        input=prompt\n    )\

## Q3. First result with vector search

In [122]:
vector_results = vector_search(q)
vector_results

[{'start': 2000,
  'content': 'wrong.\n\n## The project\n\nRAG solves these problems by giving the LLM relevant documents at\nquestion time. We don\'t hope the model memorized the answer. We\nretrieve the right information and hand it to the LLM, and the model\ngenerates a grounded response. This lets us inject knowledge the model\nnever saw during training. That\'s why RAG is still the most common way\npeople use LLMs in the industry.\n\nTo make this concrete, we build a FAQ agent for our course. A student\nasks something like "when does the course start?" and the agent answers\nfrom the FAQ data we prepared.\n\nThis module has two parts.\n\nIn Part 1 (the next 9 lessons) we will:\n\n- Understand what RAG is and how it works\n- Build a search engine over a real FAQ dataset\n- Write a prompt that combines the user\'s question with search results\n- Wire it all together into a working RAG pipeline\n- Split ingestion and query into separate processes\n\nIn Part 2, we make the pipeline ag

## Q4. Evaluating text search

In [123]:
ground_truth[0]

{'question': "What is RAG, and how does it help when an LLM doesn't know the answer by itself?",
 'document': '01-agentic-rag/lessons/01-intro.md'}

In [124]:
def hit_rate_from_textsearch_question(expected_answer, results):
    doc_id = expected_answer["document"]
    hits = sum(1 for d in results if d["filename"] == doc_id)
    return hits / len(results) if results else 0.0


def mrr_from_vectorsearch_question(expected_answer, results):
    doc_id = expected_answer["document"]
    for rank, d in enumerate(results, start=1):
        if d["filename"] == doc_id:
            return 1.0 / rank
    return 0.0


In [125]:
ground_truth[0]


{'question': "What is RAG, and how does it help when an LLM doesn't know the answer by itself?",
 'document': '01-agentic-rag/lessons/01-intro.md'}

In [126]:
hit_rate_from_textsearch_question(ground_truth[0], text_search(query=ground_truth[0]["question"]))


0.2

## Q5. Evaluating text search

In [128]:
mrr_from_vectorsearch_question(ground_truth[0], vector_search(ground_truth[0]["question"]))

1.0

## Q6. Tuning hybrid search

In [129]:
def mrr_from_search_question(expected_answer, results):
    doc_id = expected_answer["document"]
    for rank, d in enumerate(results, start=1):
        if d["filename"] == doc_id:
            return 1.0 / rank
    return 0.0

k_values = [1, 50, 100, 200]
mrrs = {}
for k in k_values:
    scores = []
    for expected in ground_truth:
        results = hybrid_search(expected["question"], k=k)
        scores.append(mrr_from_search_question(expected, results))
    mrrs[k] = sum(scores) / len(scores) if scores else 0.0

best_k = max(mrrs, key=mrrs.get)
print("MRR by k:")
for k, score in mrrs.items():
    print(f"k={k}: {score:.4f}")
print(f"Best k: {best_k} with MRR={mrrs[best_k]:.4f}")

MRR by k:
k=1: 0.6111
k=50: 0.6111
k=100: 0.6111
k=200: 0.6111
Best k: 1 with MRR=0.6111
